# Travel Itinerary Assistant: full run


In [11]:
import travel_agent.config  # loads .env and sets the API key
from agents import Runner
from travel_agent.agents.intake_agent import intake_agent
from travel_agent.pipeline import build_plan
from agents import InputGuardrailTripwireTriggered

## 1. Intake: multi-turn chat


In [12]:
history = []

async def say(msg):
    global history
    try:
        result = await Runner.run(intake_agent, history + [{"role": "user", "content": msg}])
    except InputGuardrailTripwireTriggered as exc:
        print("blocked:", exc.guardrail_result.output.output_info.reason)
        return None
    history = result.to_input_list()
    out = result.final_output
    print("missing:", out.missing_fields)
    print("assistant:", out.next_question)
    return out
out = await say("I want to visit Japan with 2 friends starting October 26 for 7 days, budget 6000 USD")
# out = await say("Write me a poem about my cat")

missing: ['trip_type', 'passport_country', 'origin']
assistant: Is this trip for leisure or business?


In [13]:
out = await say("I hold a French passport, flying from Miami. Food and temples for vacation.")

missing: []
assistant: None


In [14]:
assert out.is_complete, out.missing_fields
trip = out.trip
print(trip.model_dump_json(indent=2))

{
  "trip_type": "leisure",
  "passport_country": "France",
  "origin": "Miami",
  "destination": "Japan",
  "start_date": "2026-10-26",
  "end_date": "2026-11-01",
  "num_days": 7,
  "travelers": 3,
  "budget_usd": 6000.0,
  "interests": [
    "Food",
    "temples"
  ],
  "meeting_address": null,
  "needs_car_rental": null
}


## 2. Tool smoke test (no LLM)

Confirms mock data files and tools work before running agents.


In [15]:
from travel_agent.tools.transport_tool import search_flights_raw
from travel_agent.tools.stay_tool import search_stays_raw
from travel_agent.tools.visa_tool import check_visa_raw
from travel_agent.airports import resolve_airport

origin, dest = resolve_airport(trip.origin), resolve_airport(trip.destination)
print(origin, dest)
print((await search_flights_raw(origin, dest, trip.start_date, trip.end_date, trip.travelers, force_mock=True))["source"])
print(search_stays_raw(trip.destination, trip.num_days - 1, trip.travelers)["options"][:1])
print(check_visa_raw(trip.passport_country, trip.destination)["source"])

MIA NRT
mock
[{'name': 'Sample Tokyo City Hotel', 'area': 'Tokyo', 'total_cost_usd': 1500, 'why': 'Sample central option with convenient access to restaurants, transit, and major attractions.', 'is_estimate': True}]
orizn_api


## 3. Full pipeline

Visa, Activities, Transport and Stay run in parallel. Then budget, then Planner.


In [16]:
plan = await build_plan(trip)
print("warnings:", plan.warnings)
print(plan.budget.model_dump_json(indent=2))

warnings: []
{
  "flights_usd": 2400.0,
  "stay_usd": 1020.0,
  "activities_usd": 30.0,
  "local_transport_usd": 0.0,
  "food_usd": 945.0,
  "total_usd": 4395.0,
  "budget_usd": 6000.0,
  "within_budget": true,
  "remaining_usd": 1605.0
}


In [17]:
it = plan.itinerary
print(it.title)
print(it.summary)
print("Visa:", it.visa_note)
print("Budget:", it.budget_note)
for d in it.days:
    print(f"Day {d.day} | {d.date} | {d.city}")
    for item in d.items:
        print(" -", item)

7-Day Leisure Trip Itinerary to Kyoto, Japan
This itinerary covers 7 days in Kyoto, focusing on your interests in food and temples. You will stay at the Sample Kyoto Guesthouse for all nights. Activities are well spread out with 2 to 4 experiences daily, including visits to historic temples and enjoying local cuisine. There is some free time each day for relaxation or personal exploration. All activities and stays are based on sample and estimated data. Airport transfers are not priced in the plan. Weather during your stay is typically mild and pleasant, ideal for outdoor sightseeing and dining.
Visa: French passport holders can enter Japan without a visa for up to 90 days. This visa-free entry facilitates short-term travel to Japan. This information is from orizn_api and appears reliable, but please verify on the official Japanese government website before travel.
Budget: The estimated total cost is $4395.00 USD against your budget of $6000.00 USD, so you are within budget with $1605.